# Part 2 - Running Jupyter Notebooks on the SCC

So far in this notebook you've been working locally, or reading about commands to run on the SCC. This notebook explains how to run a **real Jupyter session directly on an SCC compute node** - including one with a GPU - so your notebooks have access to the cluster's hardware and your virtual environment.

## Option 1: SCC OnDemand (Recommended)
[SCC OnDemand](https://www.bu.edu/tech/support/research/system-usage/scc-ondemand/) is a web portal that lets you launch a Jupyter server on a compute node without manually using SSH tunnels.

**Steps:**
1. Go to `scc-ondemand.bu.edu` and log in with your BU credentials (+ Duo).
2. From the **Interactive Apps** menu, choose **Jupyter Notebook** (or **JupyterLab**).
3. Fill in the request form:
   - Number of hours
   - Number of cores
   - Whether you need a GPU (and how many)
   - Which Python module / conda environment to use
4. Click **Launch**. OnDemand queues a job for you, just like `qsub` would, and waits for a compute node to become available.
5. Once it's running, click **Connect to Jupyter** - this opens Jupyter in your browser, running on the compute node.
6. In Jupyter, select your registered kernel (e.g., "Python (energize_env)" from the previous notebook) to use your own packages.

When you're done, click **Delete** in OnDemand to end the session and free up the compute node for other users.

## Option 2: Manual SSH Tunnel (Advanced)
If you prefer the command line, you can start Jupyter yourself on a compute node and tunnel the connection back to your laptop.

**On the SCC** (after `ssh`-ing in and requesting an interactive session or submitting a batch job that starts Jupyter):
```bash
qrsh -l h_rt=02:00:00 -pe omp 4

module load python3/3.12.4
source ~/envs/energize_env/bin/activate

jupyter notebook --no-browser --port=8888 --ip=0.0.0.0
```
Note the compute node's name (e.g., `scc-xa1`) and the token printed in the output.

**On your local computer**, open a new terminal and create an SSH tunnel to that specific node through the login node:
```bash
ssh -L 8888:scc-xa1:8888 your_username@scc1.bu.edu
```

Then open `http://localhost:8888` in your local browser and paste in the token from the SCC output.

SCC OnDemand automates all of this, which is why it's the recommended approach for this workshop.

## Requesting a GPU for Your Jupyter Session
In SCC OnDemand's Jupyter launch form, there is typically a field for the number of GPUs (set it to 1 or more). If you're doing this manually with `qrsh`, add the GPU resource request:

```bash
qrsh -l h_rt=02:00:00 -l gpus=1
```

Once inside your notebook, you can verify the GPU is visible using the same check from Notebook 5:
```python
import torch
print(torch.cuda.is_available())
```

## Good Practices for Cluster-Based Notebooks
- Always specify a **time limit** appropriate to your task - an idle Jupyter session still holds a compute node (and a GPU, if requested) that other users can't use.
- Save your work and shut down the session (via OnDemand's **Delete** button, or `File > Shut Down` in Jupyter) when you're finished, rather than leaving it running.
- For long unattended runs, convert your notebook to a `.py` script (Part 1, Notebook 7) and submit it as a batch job instead of leaving an interactive notebook open.

### *Exercise*
1. Launch a Jupyter session on SCC OnDemand with 1 GPU and your `energize_env` kernel.
2. Re-run the CPU vs. GPU benchmark from Notebook 5 inside that session and confirm you get a GPU speedup.
3. Shut the session down when you're done.